# Topic: SQL: Running Total (Cumulative SUM)

## Definition
A running total (cumulative sum) calculates the accumulated value of a metric up to the current row, evaluated sequentially based on a specific order (usually date or time).

## Why Interviewers Ask This
* It is a foundational requirement for product analytics, financial dashboards, and growth tracking.
* It tests your understanding of window functions vs. standard `GROUP BY` aggregations.
* It evaluates whether you know how the default window framing works under the hood.

## Core Concepts
* **Window Function:** `SUM(column)` paired with an `OVER()` clause.
* **`ORDER BY` within `OVER()`:** Essential to define the sequence of accumulation. Without it, you get the grand total applied to every row.
* **Default Frame:** `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` is applied automatically when `ORDER BY` is used (Note: some SQL dialects default to `RANGE`, which handles ties differently).
* **`PARTITION BY`:** Used to calculate separate, independent running totals for different groups (e.g., per user, per department).

## When to Use
* Tracking cumulative revenue, signups, or sales over time.
* Calculating a running balance in a user's account or bank ledger.
* Creating "total so far" metrics for time-series data.

## Advantages
* Does not collapse rows (unlike `GROUP BY`), allowing you to keep granular transactional data alongside the running total.
* Syntax is incredibly clean and computationally efficient compared to older legacy methods like self-joins.

## Limitations
* Can be computationally expensive on massive datasets without proper indexing on the `PARTITION BY` and `ORDER BY` columns.

## Common Comparisons
* **Running Total vs. Rolling Sum:** A running total is cumulative from the very beginning (`UNBOUNDED PRECEDING`); a rolling sum looks at a fixed window (e.g., last 7 days / `7 PRECEDING`).
* **Window Function vs. `GROUP BY`:** `GROUP BY` reduces rows to a single output row per group; window functions calculate across a set of rows while keeping the original row count intact.

## Common Interview Traps
* **Forgetting `ORDER BY` inside `OVER()`:** This results in an unordered grand total appended to every row, not a cumulative running total.
* **Confusing cumulative with rolling:** Not explicitly defining the frame when asked for a "7-day rolling sum" and providing an unbounded running total instead.
* **Duplicate Dates:** If multiple transactions happen on the exact same date, relying purely on default framing can group them unexpectedly depending on if the engine defaults to `RANGE` or `ROWS`.

## SQL Syntax
```sql
SELECT 
    sale_date,
    daily_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY sale_date 
        -- Implicitly: ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM daily_revenue;
```

## 45-Second Interview Answer
"To calculate a running total, my primary approach is using the `SUM()` aggregate wrapped in an `OVER()` window function. The crucial requirement is including an `ORDER BY` clause inside the window, typically by a date or timestamp column. This creates a sequential accumulation from the first row up to the current row. If the business requires cumulative metrics per category—like running revenue per product—I simply add a `PARTITION BY` clause before the `ORDER BY` to reset the count for each group."

---

## Example Questions ans Answers

### Q1. Show cumulative signups per day for January 2025.
**Ideal Interview Answer:**
```sql
SELECT signup_date, signups,
       SUM(signups) OVER (ORDER BY signup_date) as cumulative_signups
FROM daily_signups
WHERE signup_date BETWEEN '2025-01-01' AND '2025-01-31';
```
**Common Mistakes:** Filtering *after* the window function using a CTE instead of a `WHERE` clause. If you filter first, the running total starts at 0 on Jan 1st. If you filter after, it carries over the historical totals from 2024.
**Likely Follow-up:** "The business wants to see January's data, but the running total must include historical signups from 2024. How do you adjust your query to achieve this?"

### Q2. Calculate the running total of expenses per department, ordered by date.
**Ideal Interview Answer:**
```sql
SELECT department_id, expense_date, expense_amount,
       SUM(expense_amount) OVER (PARTITION BY department_id ORDER BY expense_date) as running_expenses
FROM expenses;
```
**Common Mistakes:** Missing the `PARTITION BY` and just ordering by date, which creates a massive, company-wide running total that incorrectly mixes department expenses together.
**Likely Follow-up:** "If we add a `GROUP BY department_id` at the end of this query, what happens?"

### Q3. Show each employee's cumulative salary paid, ordered by hire date.
**Ideal Interview Answer:**
```sql
SELECT emp_id, hire_date, salary,
       SUM(salary) OVER (PARTITION BY emp_id ORDER BY hire_date) as cumulative_salary
FROM payroll;
```
**Common Mistakes:** Misunderstanding the prompt logic. By ordering by `hire_date` for payroll records, you might get weird results if an employee is paid multiple times. (Usually, payroll running totals are ordered by `pay_date`).
**Likely Follow-up:** "If an employee receives two bonus payouts on the exact same date, how does the engine decide which row gets processed first in the running total?"

### Q4. Compute the running total of page views per user, ordered by session date.
**Ideal Interview Answer:**
```sql
SELECT user_id, session_date, page_views,
       SUM(page_views) OVER (PARTITION BY user_id ORDER BY session_date) as running_views
FROM user_sessions;
```
**Common Mistakes:** Using `GROUP BY user_id` which collapses the session dates into a single row, destroying the time-series nature required for a running total.
**Likely Follow-up:** "Write the explicit frame clause (`ROWS BETWEEN...`) that the database is implicitly applying to your query."

## Practice Questions:

In [1]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

### Q1: The Duplicate Date Trap

Interviewer: "Your standard running total query looks good. However, we have a transactional table where multiple transactions can happen on the exact same day.

When you run SUM(amount) OVER (ORDER BY transaction_date), you notice that for rows with the same date, the running total isn't incrementing row-by-row. Instead, it is adding all transactions for that date together and displaying the same 'end-of-day' total for every row on that date.

Why is this happening, and how do you explicitly modify the OVER() clause to ensure the running total increments strictly row-by-row, even if the dates are identical?"

Table: transactions
- transaction_id (INT)
- transaction_date (DATE)
- amount (DECIMAL)

(Sample scenario: If two $100 transactions happen on 2025-01-01, the current query shows a running total of $200 for both rows. We want the first row to show $100, and the second to show $200).

* Answer:

```sql
SELECT transaction_id, transaction_date, amount,
       SUM(amount) OVER(
           ORDER BY transaction_date, transaction_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) as cumulative_amount
FROM transactions;
```
**Interview Tip (`ROWS` vs `RANGE`):** 
When you use `ORDER BY` in a window function, SQL implicitly applies a frame. Standard SQL engines default to `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. `RANGE` treats duplicate values in the `ORDER BY` column as "peers" and evaluates them together (grouping their totals). To force strictly row-by-row calculation, explicitly declare the frame using `ROWS` instead of `RANGE`.

**Pro-Tip (Deterministic Sorting):** Always add a unique identifier (like `transaction_id`) as a secondary `ORDER BY` condition when dates can tie. This ensures the engine sorts the tied rows in the exact same order every time the query is executed.

### Q2: The Rolling Sum

Interviewer: "Instead of a cumulative running total since the beginning of time, our marketing team wants to see a 3-day rolling sum of our revenue.

For any given transaction date, they want the sum of the revenue for that day plus the revenue from the 2 strictly preceding rows. How would you modify your window function to achieve this?"

Table: daily_revenue
- sale_date (DATE)
- daily_revenue (DECIMAL)

(Assume there is exactly one row per date).

- Answer:
```sql
SELECT sale_date, daily_revenue,
       SUM(daily_revenue) OVER(
           ORDER BY sale_date
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ) as rolling_revenue_3_day
FROM daily_revenue;
```
**Interview Tip (Running vs. Rolling):** 
To convert a cumulative running total into a rolling window, change `UNBOUNDED PRECEDING` to a specific number, like `2 PRECEDING`. Remember that a "3-day" window includes the current row PLUS the 2 preceding rows ($1 + 2 = 3$).

### Q3:The Missing Dates Trap

Interviewer: "Your ROWS BETWEEN 2 PRECEDING logic is perfect if there is exactly one row for every calendar date.

But what if our store is closed on Sundays and holidays? On those days, there is no row at all inserted into the daily_revenue table.

If Monday's row looks back 2 PRECEDING physical rows, it will grab Saturday and Friday. That's a 4-day chronological window, not a 3-day window. How can you modify your query in modern SQL (like MySQL 8.0+ or PostgreSQL) to enforce a strict 3-calendar-day rolling sum, even if there are gaps in the physical rows?"

Table: daily_revenue
- sale_date (DATE)
- daily_revenue (DECIMAL)

(Hint: Think back to the difference between ROWS and RANGE that we discussed earlier).

- Answer:
```sql
SELECT sale_date, daily_revenue,
       SUM(daily_revenue) OVER(
           ORDER BY sale_date
           RANGE BETWEEN INTERVAL 2 DAY PRECEDING AND CURRENT ROW
       ) as rolling_revenue_3_day
FROM daily_revenue;
```
**Interview Tip (`ROWS` vs. `RANGE` with Dates):** 
If a dataset has missing dates (gaps), `ROWS BETWEEN 2 PRECEDING` will pull in data from outside the 3-day chronological window because it strictly counts physical rows backward. By using `RANGE BETWEEN INTERVAL 2 DAY PRECEDING`, you instruct the engine to evaluate the logical date values. It creates a strict chronological 3-day window, correctly handling any missing days inside that window without reaching back further.